In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import skew
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.utils import resample
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from mlxtend.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
root_dir = 'C:/Users/MELİSA/smaller_dataset.csv'
df = pd.read_csv(root_dir)

In [4]:
df["Weight"] = df["Total Fwd Packet"] * df["Total Bwd packets"]

In [ ]:

rename_mapping = {
    "Benign&Bruteforce_benign": "Benign",
    "Benign&Bruteforce_BruteForce": "Brute Force"
}


df['Attack_Type'] = df['Attack_Type'].replace(rename_mapping)


In [ ]:
categorical_columns = ["Src IP", 'Dst IP', "Src Port", "Dst Port", "Protocol"]  

for col in categorical_columns:
    df[col] = df[col].astype('category')

In [7]:
for col in ['Src IP', 'Dst IP', 'Src Port', 'Dst Port']:
    df[col + '_freq'] = df[col].map(df[col].value_counts())

df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')

In [8]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'],  format='%d/%m/%Y %I:%M:%S %p')

In [9]:
attacks_to_remove = [
    "spoofing_ARP Spoofing",
    "spoofing_DNS Spoofing",
    "sqlinjection",
    "XSS",
    "Benign&Bruteforce_BruteForce",
    "Uploading_Attack"
]

df = df[~df["Attack_Type"].isin(attacks_to_remove)]

In [10]:
columns_to_drop = ["Fwd Packet Length Mean",
                   "Fwd Packet Length Min",
                   "Fwd Packet Length Std",
                   "Fwd Segment Size Avg",
                   "Total Bwd packets",
                   "Bwd Segment Size Avg",
                   "Bwd PSH Flags", 
                   "Fwd URG Flags", 
                   "Fwd IAT Min",
                   "Bwd URG Flags",
                   "Bwd Packet Length Mean",
                   "Bwd Packet Length Std",
                   "Flow IAT Min",	
                   "Flow IAT Max",
                   "Flow IAT Mean",	
                   "Flow IAT Max",    
                   "Fwd IAT Std",	
                   "Fwd IAT Max",
                   "CWR Flag Count",      
                   "Bwd IAT Min",
                   "Bwd IAT Max",
                   "Packet Length Mean",
                   "Packet Length Std",
                   "Packet Length Variance",
                   "Fwd Bytes/Bulk Avg",
                   "Fwd Packet/Bulk Avg",
                   "Fwd Bulk Rate Avg",
                   "Active Max",
                   "Active Min",
                   "Idle Max",
                   "Idle Min",
                   "Idle Std",
                   "Idle Mean",
                   "Subflow Fwd Packets",
                   "Fwd Header Length",
                   "Fwd IAT Total",
                   "Total Length of Fwd Packet",
                   "Fwd Act Data Pkts",
                   "Fwd Packet Length Max",
                   "Fwd Packets/s",
                   "Bwd Bytes/Bulk Avg",
                   "Bwd Packet/Bulk Avg",
                   "Bwd Header Length",
                   "Flow ID",
                   "URG Flag Count",
                   "Label"]

df = df.drop(columns=columns_to_drop)

In [11]:
print("Shape of the dataset:", df.shape)

print("\nColumns with Null Values:")
null_columns = df.isnull().sum()
null_columns = null_columns[null_columns > 0]  
print(null_columns)

print("\nColumns with Inf Values:")
inf_columns = df.columns[(df == np.inf).any() | (df == -np.inf).any()]
print(inf_columns)

df = df.dropna()
print("Shape of the dataset after dropping Null:", df.shape)

df = df[~df.isin([np.inf, -np.inf]).any(axis=1)]
print("Shape of the dataset after dropping Inf:", df.shape)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
print("Infs converted to NaNs.")
print(df.isnull().sum()[df.isnull().sum() > 0])



Shape of the dataset: (439194, 47)

Columns with Null Values:
Flow Bytes/s    170
dtype: int64

Columns with Inf Values:
Index(['Flow Bytes/s', 'Flow Packets/s'], dtype='object')
Shape of the dataset after dropping Null: (439024, 47)
Shape of the dataset after dropping Inf: (438920, 47)
Infs converted to NaNs.
Series([], dtype: int64)


In [ ]:

non_numeric_df = df.select_dtypes(exclude=[np.number])


non_numeric_df.head()


,Src IP,Src Port,Dst IP,Dst Port,Timestamp,Attack_Type,Proto_0,Proto_6,Proto_17
0,192.168.137.144,49153,192.168.137.240,13217,2022-08-05 10:53:38,DoS_DoS SYN Flood,False,True,False
1,192.168.137.65,7661,192.168.137.171,6668,2022-10-26 11:53:56,DDoS_DDoS ACK Fragmentation,False,True,False
2,192.168.137.12,15376,192.168.137.235,8008,2022-10-26 15:46:03,DDoS_DDoS ACK Fragmentation,False,True,False
3,192.168.137.12,21499,192.168.137.206,55442,2022-10-26 16:50:41,DDoS_DDoS ACK Fragmentation,False,True,False
4,192.168.137.225,38616,192.168.137.132,8080,2022-09-14 11:10:19,DDoS_DDoS-HTTP Flood,False,True,False


In [13]:
def create_anomaly_label(row):
    if row['Attack_Type'] == 'Benign':  
        return 'Normal'
    else:
        return 'Attack'

df['Anomaly_Label'] = df.apply(create_anomaly_label, axis=1)

print(df[['Attack_Type', 'Anomaly_Label']].head())

                   Attack_Type Anomaly_Label
0            DoS_DoS SYN Flood        Attack
1  DDoS_DDoS ACK Fragmentation        Attack
2  DDoS_DDoS ACK Fragmentation        Attack
3  DDoS_DDoS ACK Fragmentation        Attack
4         DDoS_DDoS-HTTP Flood        Attack


In [14]:
df.head()

,Src IP,Src Port,Dst IP,Dst Port,Timestamp,Flow Duration,Total Fwd Packet,Total Length of Bwd Packet,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,Flow Packets/s,Flow IAT Std,Fwd IAT Mean,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Fwd PSH Flags,Bwd Packets/s,Packet Length Min,Packet Length Max,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Bwd Bulk Rate Avg,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,FWD Init Win Bytes,Bwd Init Win Bytes,Fwd Seg Size Min,Active Mean,Active Std,Attack_Type,Weight,Src IP_freq,Dst IP_freq,Src Port_freq,Dst Port_freq,Proto_0,Proto_6,Proto_17,Anomaly_Label
0,192.168.137.144,49153,192.168.137.240,13217,2022-08-05 10:53:38,18188538,1,0.0,0.0,0.0,0.000000,0.274898,2.655540e+06,0.0,13184858.0,4.394953e+06,3.230926e+06,0,0.219919,0.0,0.0,0,4,1,0,1,0,4.0,0.0,0,0,1,0,29200,0,24,0.0,0.0,DoS_DoS SYN Flood,4,1361,1363,1417,1,False,True,False,Attack
1,192.168.137.65,7661,192.168.137.171,6668,2022-10-26 11:53:56,99736150,2,0.0,0.0,0.0,29.277248,0.020053,0.000000e+00,99736150.0,0.0,0.000000e+00,0.000000e+00,0,0.000000,1460.0,1460.0,0,0,0,0,2,0,0.0,2190.0,0,2920,0,0,512,0,20,0.0,0.0,DDoS_DDoS ACK Fragmentation,0,658,116,1,9306,False,True,False,Attack
2,192.168.137.12,15376,192.168.137.235,8008,2022-10-26 15:46:03,88834040,2,0.0,0.0,0.0,32.870283,0.022514,0.000000e+00,88834040.0,0.0,0.000000e+00,0.000000e+00,0,0.000000,1460.0,1460.0,0,0,0,0,2,0,0.0,2190.0,0,2920,0,0,512,0,20,0.0,0.0,DDoS_DDoS ACK Fragmentation,0,283,777,2,960,False,True,False,Attack
3,192.168.137.12,21499,192.168.137.206,55442,2022-10-26 16:50:41,17797062,1,0.0,0.0,0.0,82.036012,0.112378,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0,0.056189,0.0,1460.0,0,0,1,0,1,0,1.0,1460.0,0,1460,1,0,512,0,20,0.0,0.0,DDoS_DDoS ACK Fragmentation,1,283,131,1,702,False,True,False,Attack
4,192.168.137.225,38616,192.168.137.132,8080,2022-09-14 11:10:19,294063,1,0.0,0.0,0.0,0.000000,6.801264,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0,3.400632,0.0,0.0,0,1,1,0,1,0,1.0,0.0,0,0,0,0,64240,0,40,0.0,0.0,DDoS_DDoS-HTTP Flood,1,57,336,5,2296,False,True,False,Attack


In [15]:
X = df.select_dtypes(include=['number', 'bool'])
y = df['Anomaly_Label']  # Target variable
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [16]:
import multiprocessing
print(multiprocessing.cpu_count())


16


In [ ]:

from mlxtend.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
model = RandomForestClassifier(n_jobs=-1)
sfs_forward = SequentialFeatureSelector(model, k_features=25, forward=True, n_jobs=10, verbose=2).fit(X, y)


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


In [ ]:

selected_features = list(sfs_forward.k_feature_names_)

print("Selected Features:")
for i, feat in enumerate(selected_features, 1):
    print(f"{i}. {feat}")


In [ ]:

model = RandomForestClassifier(n_jobs=-1)
sfs_backward = SequentialFeatureSelector(model,
                                         k_features=25,
                                         forward=False,
                                         n_jobs=16,
                                         verbose=2).fit(X, y)


In [ ]:

selected_features = list(sfs_backward.k_feature_names_)

print("Selected Features:")
for i, feat in enumerate(selected_features, 1):
    print(f"{i}. {feat}")

In [ ]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


model = LGBMClassifier(n_jobs=-1, verbose=1, random_state=42)


rfe = RFE(estimator=model, n_features_to_select=20, step=1, verbose=1)
rfe.fit(X_train, y_train)


selected_features = X_train.columns[rfe.support_]
print("\n✅ Selected Features:", selected_features.tolist())


Fitting estimator with 41 features.
[LightGBM] [Info] Number of positive: 318558, number of negative: 32578
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019287 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7312
[LightGBM] [Info] Number of data points in the train set: 351136, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.907221 -> initscore=2.280167
[LightGBM] [Info] Start training from score 2.280167
Fitting estimator with 40 features.
[LightGBM] [Info] Number of positive: 318558, number of negative: 32578
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017520 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7310
[LightGBM] [Info] Number of data point

In [ ]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_selection import RFECV
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt



model = RandomForestClassifier(n_jobs=-1)
#model = LGBMClassifier(n_jobs=-1, verbose=1, random_state=42)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rfecv = RFECV(
    estimator=model,
    step=1,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

rfecv.fit(X_train, y_train)


selected_features = X_train.columns[rfecv.support_]
print(f"\n✅ Optimal number of features: {rfecv.n_features_}")
print(f"✅ Selected features: {selected_features.tolist()}")

Fitting estimator with 41 features.
Fitting estimator with 40 features.
Fitting estimator with 39 features.
Fitting estimator with 38 features.
Fitting estimator with 37 features.
Fitting estimator with 36 features.
Fitting estimator with 35 features.
Fitting estimator with 34 features.
Fitting estimator with 33 features.
Fitting estimator with 32 features.
Fitting estimator with 31 features.
Fitting estimator with 30 features.
Fitting estimator with 29 features.
Fitting estimator with 28 features.
Fitting estimator with 27 features.
Fitting estimator with 26 features.
Fitting estimator with 25 features.
Fitting estimator with 24 features.
Fitting estimator with 23 features.
Fitting estimator with 22 features.
Fitting estimator with 21 features.
Fitting estimator with 20 features.
Fitting estimator with 19 features.
Fitting estimator with 18 features.
Fitting estimator with 17 features.
Fitting estimator with 16 features.
Fitting estimator with 15 features.
Fitting estimator with 14 fe